# Chapter 8: Transfer Learning

This notebook accompanies **Chapter 8** of the lecture notes.

> Last lecture you trained an encoder from scratch and looked at what the bottleneck had organised. Today the encoder arrives pretrained, someone else paid for it, and the lecture is about what to do with it. Freeze it, probe it, fine-tune it. Bridge it to a second modality. Compose it with pieces it was never trained alongside. The notebook walks the same arc as the published recipe behind CLIP, SAM, and the rest. Only the model and dataset shrink so everything fits on your laptop.

**Agenda**

🧱 · 🪜 · 🎯 · 🔗 · 🧩 · 🏁

**Take it from here:** 🦙

> **Tip:** Run cells top to bottom. Later cells depend on earlier ones. Nothing here downloads weights; we pretrain a tiny backbone in seconds inside this notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys; sys.path.insert(0, '../..')
from plot_style import *
from sklearn.datasets import load_digits
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from scipy.optimize import minimize
from checks import (
    check_encode, check_linear_probe, check_attribute_table,
    check_class_centroids, check_attribute_bridge, check_cosine_classify,
    check_contrastive_loss, check_topk_retrieve,
)
from viz_helpers import (
    kshot_sweep, plot_kshot_sweep,
    train_clip_attribute_encoder, plot_similarity_before_after,
    compose_query_visualize,
)

RNG = np.random.default_rng(0)

## 🧱 Backbones

A backbone is an encoder trained on one task, then reused as a feature extractor for another. A small head on top, usually one linear layer, supplies the predictions for the new task. The backbone supplies the *representation*; the head supplies the *decision*.

The reason this works at all is the same point we made about embeddings in chapter 6. A bottlenecked encoder cannot memorise individual inputs; it has to discover regularities that hold across the training distribution. Those regularities are exactly what a downstream model needs to perform well on new, unseen data.

### A short origin story

Before backbones, vision systems were stacks of *hand-crafted features* (SIFT, HOG, Haar wavelets) designed by hand and pasted into pipelines per task. The first turning point came in 2014: two papers (Razavian et al., Yosinski et al.) showed that *off-the-shelf* features from a network pretrained on ImageNet beat the best hand-crafted features almost everywhere they were measured. ImageNet pretraining became the default starting point for vision.

The second turning point was **scale**. Once the encoders were trained on web-scale data instead of ImageNet's curated 1.4M, they stopped looking like task-specific tools and started looking like general-purpose substrates that downstream tasks plugged into. By that point the community had a new name for them, *foundation models*, and a new vocabulary (zero-shot, prompting, in-context learning) for how you actually use them. **Foundation models are backbones with three additions: scale, breadth, and (often) cross-modal alignment. No new architecture.**

| Backbone | Year | Params | Pretraining data | Transfer style |
|---|---|---|---|---|
| AlexNet / VGG / ResNet-50 | 2012 to 2015 | 60M / 138M / 25M | ImageNet, 1.4M images, 1000 classes | fine-tune the head |
| BERT-base | 2018 | 110M | Books + Wikipedia, ~3.3B tokens | fine-tune the whole model per task |
| GPT-3 | 2020 | 175B | ~500B tokens, web + books + code | in-context, no gradient updates |
| CLIP ViT-L/14 | 2021 | ~430M (vision + text) | 400M image-caption pairs from the web | zero-shot via text prompts |
| DINOv2 ViT-g | 2023 | ~1.1B | 142M curated images, no labels | linear probe on frozen features |
| SAM ViT-H | 2023 | 636M | 11M images, 1B masks | promptable segmentation, zero-shot |
| Llama-3-70B | 2024 | 70B | ~15T tokens | in-context |

These models do *exactly* what we are about to do here, only at a six-orders-of-magnitude-larger scale. The mechanic on this page is the published recipe.

### A toy backbone we can train in seconds

We pretrain on the eight digits 2 through 9, that is the "ImageNet" of this notebook. The two unseen classes 0 and 1 are held out completely; the backbone never sees a 0 or a 1 during pretraining. The whole rest of the notebook is about reaching those held-out classes through what the backbone learned on the others.

In [ ]:
digits = load_digits()
X_all, y_all = digits.data / 16.0, digits.target  # X is (1797, 64), values in [0, 1]

# Source classes (used to pretrain the backbone): digits 2..9
src_mask = (y_all >= 2)
X_src, y_src = X_all[src_mask], y_all[src_mask]

# Target classes (held out from pretraining): digits 0 and 1
tgt_mask = (y_all <= 1)
X_tgt, y_tgt = X_all[tgt_mask], y_all[tgt_mask]

print(f'Source pool : {X_src.shape}, classes {sorted(set(y_src))}')
print(f'Target pool : {X_tgt.shape}, classes {sorted(set(y_tgt))}')

In [ ]:
# Pretrain the backbone on the source classes (supervised classification).
# Hidden sizes (32, 16) means the last hidden layer is 16-dimensional;
# that 16-d activation is the embedding we will reuse downstream.
backbone = MLPClassifier(
    hidden_layer_sizes=(32, 16),
    activation='relu',
    solver='adam',
    learning_rate_init=1e-3,
    max_iter=80,
    random_state=0,
)
backbone.fit(X_src, y_src)
print(f'Pretraining accuracy on source classes: {backbone.score(X_src, y_src):.3f}')
print(f'Layer shapes (W, b):')
for i, (W, b) in enumerate(zip(backbone.coefs_, backbone.intercepts_)):
    print(f'  layer {i}: W {W.shape},  b {b.shape}')

### Strip the head and embed

The backbone has three weight matrices: 64 → 32, 32 → 16, 16 → 8. The first two are the *encoder*; the last one is the *classification head*. Transfer starts by throwing the head away and treating the 16-d activation as a general-purpose representation.

> The head was trained to separate the eight source classes. Why would the activations one layer earlier, never directly supervised, also be useful for separating digits the model has never seen?

<details><summary>Thought</summary>

Because the only way the head can do its job is if the layer below it has already organised the inputs into something close to linearly separable. A representation good enough to separate eight digits has to carry features like loops, vertical strokes, and curvature, features that are not specific to those eight classes. A 0 (round, one loop) and a 1 (vertical stroke, no loop) are described in exactly the same vocabulary the backbone already learned for 6, 8, 9, 4, and 7.
</details>

Implement `encode`. Read the trained backbone's weights, run a forward pass through the first two layers (the encoder), apply the activation, and normalise each row. The output should be `(n, 16)` with every row on the unit sphere. Useful operations: `mlp.coefs_`, `mlp.intercepts_`, `np.maximum(0, z)` for ReLU, `np.linalg.norm(..., axis=1, keepdims=True)`.

In [ ]:
def encode(X, mlp):
    """Forward-pass X through the encoder layers of mlp; return normalised embeddings."""
    W1, b1 = mlp.coefs_[0], mlp.intercepts_[0]
    W2, b2 = mlp.coefs_[1], mlp.intercepts_[1]
    h1 = np.maximum(0, X @ W1 + b1)
    Z  = np.maximum(0, h1 @ W2 + b2)
    return Z / (np.linalg.norm(Z, axis=1, keepdims=True) + 1e-12)


check_encode(encode, X_src[:50], backbone)

In [ ]:
_Z = encode(X_src, backbone)
if _Z is None:
    print('⬜ Implement encode above first.')
else:
    # PCA project to 2D for plotting (use numpy SVD; sklearn PCA would also work).
    _Zc = _Z - _Z.mean(axis=0)
    _U, _S, _Vt = np.linalg.svd(_Zc, full_matrices=False)
    _Z2 = _Zc @ _Vt[:2].T

    fig, ax = plt.subplots(figsize=(7, 6))
    _cmap = plt.get_cmap('tab10', 10)
    for d in sorted(set(y_src)):
        m = y_src == d
        ax.scatter(_Z2[m, 0], _Z2[m, 1], color=_cmap(d), s=10, linewidths=0,
                   alpha=0.7, label=str(d))
    ax.set_xlabel('PC 1'); ax.set_ylabel('PC 2')
    ax.set_title('Backbone embeddings of source classes (2..9), PCA-projected', fontsize=10, color=_GOLDEN)
    ax.legend(frameon=False, ncol=4, fontsize=8, labelcolor=_TEXT)
    tufte_axis(ax)
    plt.tight_layout()
    plt.show()

**Observe:**
- Eight clusters are visible in the 2-d shadow, even though the embedding lives in 16 dimensions. The backbone organised the source classes by identity.
- That semantic geometry is the whole asset. Everything in the rest of the notebook is a different way of cashing it in for a downstream task.

**At scale.** Run the same forward-pass-and-normalise on a frozen ResNet-50 (25M params, ImageNet pretrained) and the scatter looks similar: clusters per ImageNet class, and visually similar classes sitting next to each other (huskies near wolves, cellos near violins). Replace the backbone with DINOv2 (1.1B params, no labels, 142M images) and the clusters become semantic across far more categories than the model was ever asked to label. The PCA scatter is the same diagnostic; the substrate is what changed.

## 🪜 One-Shot with a Linear Probe

The simplest way to use a backbone is to keep it frozen and train a tiny classifier on top of its embeddings, usually a single linear layer. With one labelled example per class this is *one-shot learning*; with a handful, *few-shot*. The classifier is small, the backbone does the heavy lifting, and the only question is whether its embedding already separates the new classes.

The prediction is concrete: when target labels are scarce, the frozen probe wins because the backbone is doing most of the work; when target labels are plentiful, training from scratch eventually catches up because the model can learn task-specific features the backbone never had.

In [ ]:
# Build a target train/test split of digits 0 and 1.
# Test set is fixed; train set is sampled at different K (shots per class).
_perm = RNG.permutation(len(X_tgt))
X_tgt_p, y_tgt_p = X_tgt[_perm], y_tgt[_perm]

# Hold out 60 examples per class as the test set.
_idx_test_0 = np.where(y_tgt_p == 0)[0][:60]
_idx_test_1 = np.where(y_tgt_p == 1)[0][:60]
test_idx = np.concatenate([_idx_test_0, _idx_test_1])
pool_idx = np.array([i for i in range(len(X_tgt_p)) if i not in set(test_idx)])

X_tgt_test, y_tgt_test = X_tgt_p[test_idx], y_tgt_p[test_idx]
X_tgt_pool, y_tgt_pool = X_tgt_p[pool_idx], y_tgt_p[pool_idx]
print(f'Target test : {X_tgt_test.shape}, balanced over 0/1')
print(f'Target pool : {X_tgt_pool.shape}, available shots per class')

In [ ]:
# What does K = 1 actually look like? Pull one digit 0 and one digit 1 from the
# target pool: that pair is the entire training set in the one-shot setting.
_demo_rng = np.random.default_rng(1)
_demo_idx = [int(_demo_rng.choice(np.where(y_tgt_pool == c)[0])) for c in (0, 1)]
_X_one, _y_one = X_tgt_pool[_demo_idx], y_tgt_pool[_demo_idx]

fig, axes = plt.subplots(1, 2, figsize=(4, 2.4))
for ax, x, y in zip(axes, _X_one, _y_one):
    ax.imshow(x.reshape(8, 8), cmap='gray_r')
    ax.set_title(f'class {int(y)}', fontsize=10, color=_GOLDEN)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle('K = 1: the entire training set for the one-shot probe',
             fontsize=10, color=_TEXT)
plt.tight_layout()
plt.show()

### Linear probe on frozen embeddings

Freeze the backbone, embed the target training set, train a logistic regression on top. With K = 1 this is *one-shot*; the probe sees a single labelled 0 and a single labelled 1, and has to classify everything else.

> The probe is a single linear layer on top of a 16-d frozen embedding. With K = 5 examples per class, what limits the probe's accuracy: the size of K, or the quality of the embedding it sits on?

<details><summary>Thought</summary>

The embedding. With K = 5 the probe has barely any data to play with, but it is also a very small model, sixteen weights and a bias per class. The probe will fit in milliseconds; what determines whether it can separate the classes is whether the 16-d embedding already places 0s and 1s on different sides of a hyperplane. If the backbone learned features like "has loop" and "vertical stroke", a linear boundary separates them trivially. If it didn't, no amount of K helps.
</details>

Implement `linear_probe`. Fit a logistic regression on the embedded training set, score it on the embedded test set, and return the accuracy as a float. Useful: `LogisticRegression(max_iter=2000).fit(...).score(...)`.

In [ ]:
def linear_probe(emb_train, y_train, emb_test, y_test):
    """Fit logistic regression on (emb_train, y_train), return test accuracy."""
    clf = LogisticRegression(max_iter=2000)
    clf.fit(emb_train, y_train)
    return float(clf.score(emb_test, y_test))


_rng = np.random.default_rng(1)
def _k_shot(K, rng=_rng):
    idx = []
    for c in (0, 1):
        ci = np.where(y_tgt_pool == c)[0]
        idx.extend(rng.choice(ci, size=K, replace=False))
    idx = np.array(idx)
    return X_tgt_pool[idx], y_tgt_pool[idx]

_Xs, _ys = _k_shot(5)
_emb_train = encode(_Xs, backbone)
_emb_test  = encode(X_tgt_test, backbone)
check_linear_probe(linear_probe, _emb_train, _ys, _emb_test, y_tgt_test)

In [ ]:
# K-shot sweep: frozen+probe vs train-from-scratch on raw pixels.
# Both classifiers see the same (X_train, y_train); the difference is whether
# they sit on top of the backbone's embedding or on the raw 64-d pixel input.
if linear_probe(_emb_train, _ys, _emb_test, y_tgt_test) is None:
    print('⬜ Implement linear_probe above first.')
else:
    _Ks = [1, 2, 5, 10, 20, 50]
    _acc_probe, _acc_scratch = kshot_sweep(
        encode, linear_probe, backbone,
        X_tgt_pool, y_tgt_pool, X_tgt_test, y_tgt_test,
        Ks=_Ks,
    )
    plot_kshot_sweep(_Ks, _acc_probe, _acc_scratch)

In [ ]:
# Where do the held-out 0s and 1s actually fall in the embedding space the
# backbone learned from digits 2..9? Project all source embeddings and all
# held-out target embeddings into the same 2D PCA frame (fit on the source)
# and overlay them. The backbone never saw a 0 or a 1 during pretraining,
# yet they still land in coherent clusters because the features the
# backbone learned (loops, vertical strokes, curvature) describe 0s and 1s
# too. That is exactly why the linear probe at low K works.
_Z_src = encode(X_src, backbone)
_Z_tgt = encode(X_tgt, backbone)
if _Z_src is None or _Z_tgt is None:
    print('\u2b1c Implement encode above first.')
else:
    _mu = _Z_src.mean(axis=0)
    _Vt = np.linalg.svd(_Z_src - _mu, full_matrices=False)[2]
    _proj = lambda Z: (Z - _mu) @ _Vt[:2].T
    _src_2d = _proj(_Z_src)
    _tgt_2d = _proj(_Z_tgt)

    fig, ax = plt.subplots(figsize=(7.5, 6))
    _cmap = plt.get_cmap('tab10', 10)

    # Source classes (2..9): faded background.
    for d in sorted(set(y_src)):
        m = y_src == d
        ax.scatter(_src_2d[m, 0], _src_2d[m, 1], color=_cmap(d), s=8,
                   alpha=0.22, linewidths=0, label=f'source {d}')

    # Held-out target classes (0, 1): larger, with edges.
    for d, marker in [(0, 'o'), (1, '^')]:
        m = y_tgt == d
        ax.scatter(_tgt_2d[m, 0], _tgt_2d[m, 1], color=_cmap(d), s=34,
                   marker=marker, alpha=0.85, edgecolor='black',
                   linewidths=0.5, label=f'held-out {d}', zorder=3)

    ax.set_xlabel('PC 1'); ax.set_ylabel('PC 2')
    ax.set_title('Held-out 0s and 1s overlaid on the source embedding \u2014 '
                 'backbone never saw 0 or 1', fontsize=10, color=_GOLDEN)
    ax.legend(frameon=False, fontsize=8, labelcolor=_TEXT, ncol=2)
    tufte_axis(ax)
    plt.tight_layout()
    plt.show()

**Observe:**
- Both lines reach high accuracy quickly because 0 vs 1 on raw 8x8 pixels is a very easy two-class problem. On this toy dataset the from-scratch MLP keeps up with, and at low K even slightly leads, the frozen probe.
- The expected gap (frozen probe leading at low K) shows up when the target task is harder: more classes, less separable inputs, or harder visual structure than 0 vs 1. The mechanic is the same; the toy task is just too simple for the embedding's advantage to dominate.
- Small target dataset, similar domain → freeze the backbone. Large target dataset, different domain → fine-tuning becomes worth the cost. The choice is how much you trust the backbone's features.

## 🎯 Lampert: Attribute-Based Zero-Shot

The probe still needed a few labelled examples of 0 and 1. *Zero-shot* asks for none. Lampert's 2009 idea was to bridge images to unseen classes through a hand-built attribute table: every class, seen or unseen, is described by the same fixed vocabulary of attributes, and a classifier learned on seen classes carries over because the bridge is shared.

We use the same recipe. The attributes are five binary features that distinguish digits visually: presence of a closed loop, presence of two loops, a strong vertical stroke, a strong horizontal stroke, and overall curviness. The source rows (digits 2 through 9) are already filled in below. Your job is the two target rows, 0 and 1, which is exactly what a Lampert-era researcher would have done by hand to define a new class.

> Why does a class description in the same attribute vocabulary as the seen classes give a working classifier for an unseen class, even though the model has never seen one?

<details><summary>Thought</summary>

Because the bridge between attributes and embeddings was learned on the seen classes, and that bridge is what gets reused. If the source rows tell us that "has loop" maps consistently to a particular direction in embedding space, and "vertical stroke" to a different direction, then any new class described in those same coordinates gets a predicted location in embedding space without ever being seen. Classification is then nearest-neighbour against that prediction.
</details>

In [ ]:
# Attribute names: the shared vocabulary.
attr_names = ['has_loop', 'two_loops', 'vertical_stroke', 'horizontal_stroke', 'curvy']

# Source-class attributes: digits 2..9. These are pre-filled by the instructor;
# students are not expected to argue with them in this exercise.
attrs_source = {
    2: [0, 0, 0, 1, 1],   # curvy with a horizontal bar at the bottom
    3: [0, 0, 0, 0, 1],   # all curves, no loops, no straight strokes
    4: [0, 0, 1, 1, 0],   # vertical and horizontal strokes
    5: [0, 0, 0, 1, 1],   # curve plus a horizontal stroke at the top
    6: [1, 0, 0, 0, 1],   # one loop at the bottom, curvy
    7: [0, 0, 1, 1, 0],   # diagonal/vertical stroke and a horizontal top
    8: [1, 1, 0, 0, 1],   # two loops, fully curvy
    9: [1, 0, 1, 0, 1],   # one loop on top with a tail going down
}

# Confirm the source table is well-formed.
for d, row in attrs_source.items():
    assert len(row) == len(attr_names)
print('Source attribute table:')
for d in sorted(attrs_source):
    print(f'  {d}: ' + '  '.join(f'{n}={v}' for n, v in zip(attr_names, attrs_source[d])))

### Fill in the unseen-class rows

Define `attrs_target` as a dict mapping each held-out digit (`0` and `1`) to a five-element list of 0s and 1s, in the same order as `attr_names`. Think of how each digit looks: a 0 is a single closed loop with no straight strokes; a 1 is a single vertical stroke with no loops.

The checker validates the format and the rough shape of each row, not the *exact* answer. Small disagreements are fine.

In [ ]:
attrs_target = {
    # Order matches attr_names: [has_loop, two_loops, vertical_stroke, horizontal_stroke, curvy]
    # 0 is round, has one closed loop, no straight strokes, fully curvy.
    0: [1, 0, 0, 0, 1],
    # 1 is dominated by a vertical stroke, no loops, no curves, no horizontal bar.
    1: [0, 0, 1, 0, 0],
}


check_attribute_table(attrs_target, attr_names)

### Class centroids in embedding space

The bridge needs anchor points. For each source class, compute the mean of its embeddings, the *prototype* of that class. The attribute-to-embedding map will be fitted against these prototypes.

In [ ]:
def class_centroids(emb, labels):
    """Return a dict {class_label: centroid_vector} of normalised mean embeddings."""
    out = {}
    for label in sorted(set(labels.tolist() if hasattr(labels, 'tolist') else labels)):
        mask = (np.asarray(labels) == label)
        c = emb[mask].mean(axis=0)
        out[int(label)] = c / (np.linalg.norm(c) + 1e-12)
    return out


_emb_src = encode(X_src, backbone)
check_class_centroids(class_centroids, _emb_src, y_src)

### Attribute-to-embedding bridge

Fit a linear map W from attribute vectors to embedding centroids using the source classes. Closed form: stack the source attribute rows into a matrix A (shape `n_src_classes × n_attrs`) and the corresponding centroids into a matrix C (shape `n_src_classes × 16`). Solve `A @ W = C` for W in the least-squares sense. Then apply W to the unseen-class attribute rows to predict their centroids.

> The eight source rows give us eight equations in five unknowns per output dimension. Why does that work, and what would happen if we had only three source classes?

<details><summary>Thought</summary>

Eight equations in five unknowns is overdetermined per output dimension, so least-squares finds the W that best fits all of them simultaneously: generalisation by averaging. Three source classes would leave the system underdetermined (three equations, five unknowns), so W could fit the source perfectly while behaving arbitrarily on unseen attribute combinations. Source breadth is the silent ingredient that makes attribute-based zero-shot work.
</details>

In [ ]:
def attribute_bridge(attrs_source, centroids_source, attrs_target):
    """Fit W via least squares on source pairs, apply to target attribute rows."""
    src_keys = sorted(attrs_source.keys())
    A = np.array([attrs_source[k] for k in src_keys], dtype=np.float64)
    C = np.array([np.asarray(centroids_source[k]) for k in src_keys], dtype=np.float64)
    W, *_ = np.linalg.lstsq(A, C, rcond=None)
    out = {}
    for label, attr_row in attrs_target.items():
        pred = np.asarray(attr_row, dtype=np.float64) @ W
        out[label] = pred / (np.linalg.norm(pred) + 1e-12)
    return out


_centroids_src = class_centroids(_emb_src, y_src)
if _centroids_src is not None and attrs_target.get(0) is not None and attrs_target.get(1) is not None:
    check_attribute_bridge(attribute_bridge, attrs_source, _centroids_src, attrs_target)
else:
    print('⬜ Need encode, attrs_target, and class_centroids implemented first.')

### Cosine classify and read the score

With predicted centroids for 0 and 1 in hand, classify each test image by which centroid its embedding is closest to under cosine similarity. Both embeddings and centroids are unit-norm, so cosine similarity is just a dot product.

In [ ]:
def cosine_classify(query_emb, class_emb_dict):
    """Return predicted labels by argmax cosine similarity."""
    labels = sorted(class_emb_dict.keys())
    M = np.stack([np.asarray(class_emb_dict[l]) for l in labels])
    S = query_emb @ M.T
    return np.array([labels[i] for i in np.argmax(S, axis=1)])


_q = np.array([[1.0, 0.0], [0.0, 1.0]])
_c = {'a': np.array([1.0, 0.0]), 'b': np.array([0.0, 1.0])}
check_cosine_classify(cosine_classify, _q, _c, expected=np.array(['a', 'b']))

In [ ]:
# Run zero-shot on the held-out target set.
_emb_tgt_test = encode(X_tgt_test, backbone)
_centroids_src = class_centroids(_emb_src, y_src)
_predicted_tgt = attribute_bridge(attrs_source, _centroids_src, attrs_target) if _centroids_src is not None else None

if _predicted_tgt is None:
    print('⬜ Need attribute_bridge implemented first.')
else:
    _preds = cosine_classify(_emb_tgt_test, _predicted_tgt)
    if _preds is None:
        print('⬜ Need cosine_classify implemented first.')
    else:
        _acc = float((_preds == y_tgt_test).mean())
        print(f'Lampert zero-shot accuracy on (0, 1): {_acc:.3f}')

        # Confusion matrix.
        _cm = np.zeros((2, 2), dtype=int)
        for t, p in zip(y_tgt_test, _preds):
            _cm[int(t), int(p)] += 1
        print(f'Confusion matrix [rows=true, cols=pred]:')
        print(f'         pred=0  pred=1')
        print(f' true=0  {_cm[0,0]:>6}  {_cm[0,1]:>6}')
        print(f' true=1  {_cm[1,0]:>6}  {_cm[1,1]:>6}')

In [ ]:
# What does the Lampert pipeline look like on a single test image? Show one
# held-out 0 and one held-out 1, each next to its filled-in attribute caption,
# and the cosine similarity to both bridge-predicted centroids. The first
# number is the score against the centroid the bridge predicted for class 0,
# the second against the centroid for class 1; whichever is larger wins.
if _predicted_tgt is None or attrs_target.get(0) is None or attrs_target.get(1) is None:
    print('\u2b1c Need attrs_target and predicted centroids first.')
else:
    fig, axes = plt.subplots(2, 2, figsize=(8, 4.5),
                             gridspec_kw={'width_ratios': [1, 2]})
    for row, demo in enumerate([0, 1]):
        _idx = int(np.where(y_tgt_test == demo)[0][0])
        _img_emb = _emb_tgt_test[_idx]
        _attr_row = attrs_target[demo]
        _caption = ',  '.join(f'{n}={int(v)}' for n, v in zip(attr_names, _attr_row))
        _cos_0 = float(_img_emb @ np.asarray(_predicted_tgt[0]))
        _cos_1 = float(_img_emb @ np.asarray(_predicted_tgt[1]))

        ax_img, ax_txt = axes[row]
        ax_img.imshow(X_tgt_test[_idx].reshape(8, 8), cmap='gray_r')
        ax_img.set_xticks([]); ax_img.set_yticks([])
        ax_img.set_title(f'image \u2014 true digit {demo}', fontsize=10, color=_GOLDEN)

        ax_txt.axis('off')
        ax_txt.text(0.0, 0.88, f'caption (attrs_target[{demo}]):', fontsize=10,
                    color=_TEXT, transform=ax_txt.transAxes)
        ax_txt.text(0.0, 0.66, _caption, fontsize=9, color=_GOLDEN,
                    family='monospace', transform=ax_txt.transAxes)
        ax_txt.text(0.0, 0.34, f'cos(image, pred-0) : {_cos_0:+.2f}', fontsize=9,
                    color=_TEXT, family='monospace', transform=ax_txt.transAxes)
        ax_txt.text(0.0, 0.14, f'cos(image, pred-1) : {_cos_1:+.2f}', fontsize=9,
                    color=_TEXT, family='monospace', transform=ax_txt.transAxes)
    plt.tight_layout()
    plt.show()

In [ ]:
# PCA-project the 16-d embedding to 2D and overlay everything: source-class
# clouds, source centroids, predicted target centroids (stars), and the real
# held-out test images. Each test image is *coloured* by which predicted
# centroid it is closer to under cosine similarity (the actual assignment),
# and *shaped* by its true class. Mismatches between colour and shape are
# misclassifications.
if _centroids_src is None or _predicted_tgt is None:
    print('\u2b1c Need centroids and predicted target centroids first.')
else:
    _mu = _emb_src.mean(axis=0)
    _U, _S, _Vt = np.linalg.svd(_emb_src - _mu, full_matrices=False)
    _proj = lambda Z: (np.atleast_2d(Z) - _mu) @ _Vt[:2].T

    _src_2d = _proj(_emb_src)
    _tgt_test_2d = _proj(_emb_tgt_test)
    _src_cents_2d = {k: _proj(v)[0] for k, v in _centroids_src.items()}
    _tgt_cents_2d = {k: _proj(v)[0] for k, v in _predicted_tgt.items()}

    # Cosine-based assignment in the original 16-d space.
    _c0 = np.asarray(_predicted_tgt[0]); _c1 = np.asarray(_predicted_tgt[1])
    _sim0 = _emb_tgt_test @ _c0
    _sim1 = _emb_tgt_test @ _c1
    _pred = np.where(_sim0 >= _sim1, 0, 1)

    fig, ax = plt.subplots(figsize=(8.5, 6.5))
    _cmap = plt.get_cmap('tab10', 10)

    # Faded source clouds.
    for d in sorted(set(y_src)):
        m = y_src == d
        ax.scatter(_src_2d[m, 0], _src_2d[m, 1], color=_cmap(d),
                   s=8, alpha=0.15, linewidths=0)

    # Source centroids: labelled circles.
    for d, c in _src_cents_2d.items():
        ax.scatter(c[0], c[1], color=_cmap(d), s=170,
                   edgecolor='black', linewidths=1.0, zorder=3)
        ax.annotate(str(d), (c[0], c[1]), color='white', fontsize=10,
                    ha='center', va='center', fontweight='bold', zorder=4)

    # Test images: shape = true class, colour = predicted class.
    for true_label, marker, name in [(0, 'o', 'true 0'), (1, '^', 'true 1')]:
        m = y_tgt_test == true_label
        for pred_label in (0, 1):
            sub = m & (_pred == pred_label)
            if sub.any():
                ax.scatter(_tgt_test_2d[sub, 0], _tgt_test_2d[sub, 1],
                           color=_cmap(pred_label), marker=marker, s=34,
                           alpha=0.75, edgecolor='black', linewidths=0.4,
                           label=f'{name} \u2192 assigned {pred_label}')

    # Predicted target centroids: stars.
    for d, c in _tgt_cents_2d.items():
        ax.scatter(c[0], c[1], color=_cmap(d), s=320, marker='*',
                   edgecolor='black', linewidths=1.3, zorder=5,
                   label=f'predicted centroid for {d}')

    ax.set_xlabel('PC 1'); ax.set_ylabel('PC 2')
    ax.set_title('Lampert in embedding space \u2014 colour = assigned class '
                 '(cosine), shape = true class', fontsize=10, color=_GOLDEN)
    ax.legend(frameon=False, fontsize=8, labelcolor=_TEXT, loc='best')
    tufte_axis(ax)
    plt.tight_layout()
    plt.show()

In [ ]:
# Decision margin per test image: cos(image, predicted-0) - cos(image, predicted-1).
# Positive margin classifies as 0, negative as 1. Splitting the histogram by
# the *true* class shows how cleanly each side of the decision is recovered.
if _predicted_tgt is None:
    print('\u2b1c Need predicted centroids first.')
else:
    _c0 = np.asarray(_predicted_tgt[0])
    _c1 = np.asarray(_predicted_tgt[1])
    _margin = _emb_tgt_test @ _c0 - _emb_tgt_test @ _c1

    fig, ax = plt.subplots(figsize=(7, 3.2))
    _cmap = plt.get_cmap('tab10', 10)
    _bins = np.linspace(_margin.min(), _margin.max(), 24)
    for d in (0, 1):
        m = y_tgt_test == d
        ax.hist(_margin[m], bins=_bins, alpha=0.6, color=_cmap(d),
                label=f'true class {d}', edgecolor='black', linewidth=0.4)
    ax.axvline(0, color=_TEXT, linestyle='--', linewidth=0.8)
    ax.set_xlabel('cos(image, pred-0) \u2212 cos(image, pred-1)')
    ax.set_ylabel('count')
    ax.set_title('Decision margin: positive \u2192 classified as 0',
                 fontsize=10, color=_GOLDEN)
    ax.legend(frameon=False, labelcolor=_TEXT)
    tufte_axis(ax)
    plt.tight_layout()
    plt.show()

**Observe:**
- The model classifies digits it was never trained on, using only an attribute description as the bridge. No labelled 0s, no labelled 1s.
- The confusion matrix is uneven: the 0 class is recovered cleanly while many 1s get mistaken for 0s. The attribute row for 1 (vertical stroke only) does not match any source row exactly; every source class with `vertical_stroke=1` also has `horizontal_stroke=1`. The bridge has no clean direction for "vertical only", so 1s drift toward the nearest available prototype.
- Lampert is the historical answer to "how do you reach unseen classes". The next section asks the model to learn the bridge for itself.

**At scale.** Lampert's original 2009 paper bridged 50 animal classes through 85 attributes (`has_horns`, `lives_in_water`, `is_brown`) hand-curated by the authors. That handcrafting cost is exactly what makes the approach unscalable: every new domain needs a new attribute vocabulary built by experts. Replacing the attribute table with **natural language**, and the least-squares fit with **contrastive training on web-scale image-text pairs**, is the move that gets you CLIP. Same picture, different bridge.

## 🔗 Mini-CLIP: Contrastive Cross-Modal Alignment

CLIP replaces the hand-built attribute bridge with a learned one. Two encoders, one for images and one for text, are trained jointly so that an image and its matching caption land on the same point in a shared space, and unrelated pairs are pushed apart. The training signal is a single matrix: pair up N images with N captions, compute the N×N similarity matrix, and ask the model to make the diagonal big and everything else small.

We do that here with the same backbone embeddings (image side) and the same attribute vectors (taking the place of text). The image encoder is frozen; we already have the embeddings. The "text" encoder is a single linear layer mapping attribute vectors to the same 16-d space.

In [ ]:
# What does the training data actually look like? CLIP trains on (image,
# caption) pairs. Our 'captions' are the attribute rows; here are four
# source-class images shown next to their captions (only the attributes
# that are 1 — read like a short English description of the digit).
_demo_classes = [4, 6, 8, 9]
fig, axes = plt.subplots(1, len(_demo_classes), figsize=(9, 2.8))
for ax, d in zip(axes, _demo_classes):
    _idx = int(np.where(y_src == d)[0][0])
    ax.imshow(X_src[_idx].reshape(8, 8), cmap='gray_r')
    ax.set_xticks([]); ax.set_yticks([])
    _on = [n for n, v in zip(attr_names, attrs_source[d]) if v]
    _caption = '\n'.join(_on) if _on else '\u2014'
    ax.set_title(f'digit {d}', fontsize=10, color=_GOLDEN)
    ax.text(0.5, -0.18, _caption, transform=ax.transAxes, ha='center',
            va='top', fontsize=8, color=_TEXT, family='monospace')
fig.suptitle('Sample (image, caption) pairs the bridge will be trained on',
             fontsize=10, color=_TEXT)
plt.tight_layout()
plt.show()

### The diagonal contrastive loss

The InfoNCE / CLIP loss takes a batch of N image embeddings and N text embeddings (paired by index), forms an N×N similarity matrix scaled by a temperature, and applies cross-entropy in both directions, image-to-text and text-to-image, encouraging the diagonal to dominate each row and each column.

Implement `contrastive_loss(img_embs, attr_embs, tau)`. Both inputs are `(N, d)` and assumed normalised. Compute `S = (img_embs @ attr_embs.T) / tau`, treat each row as logits over text candidates with the correct answer being the diagonal index, compute cross-entropy, do the same column-wise, and return their average.

Useful operations: `np.log(np.exp(s).sum(axis=1))` is the log-sum-exp; subtract `max` first for numerical stability. The cross-entropy of row `i` is `-S[i, i] + logsumexp(S[i, :])`.

In [ ]:
def contrastive_loss(img_embs, attr_embs, tau=0.1):
    """Symmetric InfoNCE loss on (N, d) paired embeddings."""
    S = (img_embs @ attr_embs.T) / tau
    m_r = S.max(axis=1, keepdims=True)
    lse_r = np.log(np.exp(S - m_r).sum(axis=1)) + m_r.ravel()
    m_c = S.max(axis=0, keepdims=True)
    lse_c = np.log(np.exp(S - m_c).sum(axis=0)) + m_c.ravel()
    diag = np.diag(S)
    loss_i2t = (lse_r - diag).mean()
    loss_t2i = (lse_c - diag).mean()
    return float(0.5 * (loss_i2t + loss_t2i))


_rng = np.random.default_rng(0)
_v = _rng.normal(size=(8, 16))
_v = _v / np.linalg.norm(_v, axis=1, keepdims=True)
check_contrastive_loss(contrastive_loss, _v)

In [ ]:
# Train the attribute encoder W (shape (n_attrs, d_emb)) by minimising the
# contrastive loss on pairs (image_embedding, attribute_vector_of_its_class).
# Free parameters: 5 attributes * 16 dim = 80, small enough for L-BFGS-B.
_emb_src = encode(X_src, backbone)
_A_train = np.array([attrs_source[int(y)] for y in y_src], dtype=np.float64)

W_attr, _w0, _idx_pairs = train_clip_attribute_encoder(
    contrastive_loss, attrs_source, _emb_src, y_src, attr_names,
    n_pairs=300, tau=0.1, maxiter=60, seed=0,
)

### Reading the similarity matrix

The next plot is an 8×8 matrix `S` computed on a diagnostic batch — one image and one caption per source class (digits 2..9).

- **Rows** are images, embedded by the frozen backbone into 16-d space.
- **Columns** are captions (attribute rows), pushed through the bridge `W` into the *same* 16-d space.
- Both sides are normalised to unit length, so each cell `S[i, j]` is the **cosine similarity** between image `i` and caption `j` — equivalently, their dot product. Range: −1 (opposite directions) to +1 (same direction).

The matched pair for each class sits on the diagonal (`S[i, i]`); off-diagonal cells are mismatched image-caption pairs. A perfectly trained bridge would have a bright diagonal and a dim background.

In [ ]:
# Visualise the similarity matrix S on an 8x8 diagnostic batch (one image and
# one caption per source class). Both sides are unit-normalised in the shared
# 16-d embedding space, so every cell is a cosine similarity (= dot product),
# range [-1, 1]. Diagonal cells are matched pairs; off-diagonal cells are
# mismatched. Training should brighten the diagonal and dim the rest.
if W_attr is None:
    print('⬜ Train W_attr above first.')
else:
    plot_similarity_before_after(_emb_src, _A_train, _w0, W_attr, y_src)

In [ ]:
# Make the image-caption pairing concrete: pick one source-class image and
# its attribute row (the 'caption'), and look at the cosine similarity
# between the two sides before vs after training the bridge.
if W_attr is None:
    print('⬜ Train W_attr above first.')
else:
    _demo_class = 8
    _idx_demo = int(np.where(y_src == _demo_class)[0][0])
    _attr_row = np.asarray(attrs_source[_demo_class], dtype=np.float64)
    _caption = ',  '.join(f'{n}={int(v)}' for n, v in zip(attr_names, _attr_row))

    # _w0 is the flat init vector L-BFGS-B optimised; reshape to (n_attrs, d_emb)
    # to match W_attr.
    _W_before = _w0.reshape(W_attr.shape)

    # Both sides through their encoders into the shared 16-d space.
    _img_emb = encode(X_src[_idx_demo:_idx_demo + 1], backbone)[0]
    _text_before = _attr_row @ _W_before
    _text_before = _text_before / (np.linalg.norm(_text_before) + 1e-12)
    _text_after = _attr_row @ W_attr
    _text_after = _text_after / (np.linalg.norm(_text_after) + 1e-12)
    _cos_before = float(_img_emb @ _text_before)
    _cos_after = float(_img_emb @ _text_after)

    fig, (ax_img, ax_txt) = plt.subplots(
        1, 2, figsize=(7, 2.6), gridspec_kw={'width_ratios': [1, 2]})
    ax_img.imshow(X_src[_idx_demo].reshape(8, 8), cmap='gray_r')
    ax_img.set_xticks([]); ax_img.set_yticks([])
    ax_img.set_title(f'image — digit {_demo_class}', fontsize=10, color=_GOLDEN)

    ax_txt.axis('off')
    ax_txt.text(0.0, 0.88, 'caption (attribute row):', fontsize=10, color=_TEXT,
                transform=ax_txt.transAxes)
    ax_txt.text(0.0, 0.66, _caption, fontsize=9, color=_GOLDEN,
                family='monospace', transform=ax_txt.transAxes)
    ax_txt.text(0.0, 0.32, f'cosine(image, caption) before training : {_cos_before:+.2f}',
                fontsize=9, color=_TEXT, family='monospace', transform=ax_txt.transAxes)
    ax_txt.text(0.0, 0.12, f'cosine(image, caption) after  training : {_cos_after:+.2f}',
                fontsize=9, color=_TEXT, family='monospace', transform=ax_txt.transAxes)
    plt.tight_layout()
    plt.show()

In [ ]:
# Re-run zero-shot on (0, 1) using the LEARNED attribute encoder.
if W_attr is None:
    print('⬜ Train W_attr above first.')
elif attrs_target.get(0) is None or attrs_target.get(1) is None:
    print('⬜ Fill in attrs_target rows above first.')
else:
    _A_tgt = np.array([attrs_target[c] for c in (0, 1)], dtype=np.float64)
    _txt_tgt = _A_tgt @ W_attr
    _txt_tgt = _txt_tgt / (np.linalg.norm(_txt_tgt, axis=1, keepdims=True) + 1e-8)
    _learned_centroids = {0: _txt_tgt[0], 1: _txt_tgt[1]}

    _preds_clip = cosine_classify(_emb_tgt_test, _learned_centroids)
    if _preds_clip is None:
        print('⬜ Need cosine_classify implemented first.')
    else:
        _acc_clip = float((_preds_clip == y_tgt_test).mean())
        # Recompute Lampert baseline for direct comparison.
        _predicted_tgt = attribute_bridge(attrs_source, class_centroids(_emb_src, y_src), attrs_target)
        _acc_lampert = float((cosine_classify(_emb_tgt_test, _predicted_tgt) == y_tgt_test).mean())
        print(f'Lampert  (least-squares bridge)     : {_acc_lampert:.3f}')
        print(f'Mini-CLIP (contrastive bridge)      : {_acc_clip:.3f}')

**Observe:**
- The before-training matrix is uniformly noisy. After training, the values vary more: some diagonal cells are bright, others are not, and a handful of off-diagonal cells stay warm. Eighty parameters and sixty L-BFGS-B iterations buy a partial alignment, not the clean lit-diagonal you would see after long training on a large image-text corpus.
- Both bridges use the same source data and the same attribute table; only the *fitting criterion* differs. Least-squares minimises a regression error; contrastive minimises a discrimination error. The discrimination error is closer to what we actually use the bridge for at test time, and the (0, 1) accuracy goes up accordingly.

This is the published CLIP recipe (same loss, same paired encoders, same normalised embeddings) at six orders of magnitude smaller scale.

## 🧩 Composing the Pieces

Three things were built above: a backbone (image encoder), an attribute encoder (the "text" side of mini-CLIP), and a similarity-based classifier. None of them was trained for the task we are about to give them: *retrieve every digit in the dataset that matches a free-form attribute query, then describe what you found*.

We compose: the attribute query goes through the trained text encoder; cosine ranks every image in the dataset; the top matches are clustered with k-means (from the lecture-02 notebook); cluster centroids reveal what the retrieved set has in common. None of those steps required new training.

### Top-K retrieval

Implement `topk_retrieve(query_emb, all_embs, k)`. Return the indices of the k rows of `all_embs` most similar to `query_emb` under cosine similarity, in descending order. Both inputs are normalised so cosine is a dot product. Useful: `np.argsort(...)`.

In [ ]:
def topk_retrieve(query_emb, all_embs, k):
    """Return indices of the k rows of all_embs most similar to query_emb."""
    sims = all_embs @ query_emb
    order = np.argsort(sims)[::-1]
    return order[:k]


_qq = np.array([1.0, 0.0])
_aa = np.array([[0.0, 1.0], [0.9, 0.1], [1.0, 0.0], [-1.0, 0.0]])
_aa = _aa / np.linalg.norm(_aa, axis=1, keepdims=True)
check_topk_retrieve(topk_retrieve, _qq, _aa, k=2, expected=np.array([2, 1]))

In [ ]:
# Compose the pieces: an attribute query none of the components was trained for.
# Query: "round, with a closed loop, no straight strokes". That is the canonical
# description of a 0, but the contrastive bridge was trained only on classes
# 2..9, so the digit 0 was never paired with this attribute vector during
# training.
if W_attr is None:
    print('⬜ Train W_attr above first.')
else:
    compose_query_visualize(
        W_attr, encode, topk_retrieve, X_all, y_all, attr_names,
        query_attr=np.array([1, 0, 0, 0, 1], dtype=np.float64),
        backbone=backbone,
    )

In [ ]:
# A second retrieval: different attribute query, different cluster. The query
# [1, 1, 0, 0, 1] = 'has_loop, two_loops, curvy' is unique to digit 8 in the
# source table (two_loops=1 only matches 8), so the bridge should pull 8s
# rather than 6s.
if W_attr is None:
    print('⬜ Train W_attr above first.')
else:
    compose_query_visualize(
        W_attr, encode, topk_retrieve, X_all, y_all, attr_names,
        query_attr=np.array([1, 1, 0, 0, 1], dtype=np.float64),
        backbone=backbone,
    )

**Observe:**
- The top retrievals are all 6s, not 0s. The query attribute row `[1, 0, 0, 0, 1]` matches both the held-out 0 and the source class 6, but only 6 was paired with this row during training, so 6's neighbourhood is what the bridge has learned to project onto.
- The composition still does what it advertised mechanically: an attribute query, a frozen backbone, a learned bridge, and a clustering algorithm assembled at inference time, with no new training. The result here also makes the limitation legible: zero-shot retrieval of a held-out class needs an attribute pattern that distinguishes it from every source class, and our toy table does not.

The pattern is what drives most foundation-model deployments: pretrain large pieces on broad data, then *compose* them at inference time for tasks none of them was built for. Chapter 14 picks the thread back up and scales it to embodied action with Vision-Language-Action models that compose perception, language, and motor control.

### 🏁 Recap

**What we did:**
- 🧱 Pretrained a tiny MLP backbone on digits 2..9 and stripped the head to use the 16-d hidden activation as a general-purpose embedding. Walked through the historical line from hand-crafted features through ImageNet pretraining to web-scale foundation models.
- 🪜 Trained a linear probe on top of the frozen backbone and watched it dominate from K = 1 (one-shot) upward, while an MLP trained from scratch on the same target labels stayed near chance until K was large.
- 🎯 Reached unseen classes (0 and 1) through Lampert's attribute bridge, fitting a least-squares map from a hand-built attribute table to embedding centroids.
- 🔗 Replaced the hand-built bridge with a learned one via the diagonal contrastive loss (mini-CLIP) and saw the similarity matrix shift from noise to a lit diagonal.
- 🧩 Composed the trained pieces into a free-form attribute query → retrieval → cluster pipeline that none of them was individually trained for.

**Key takeaways:**
- A foundation model is a backbone with three additions: scale, breadth, and (often) cross-modal alignment. The architecture is not new; the data and the parameter count are.
- A frozen backbone with a linear probe on top is the cheapest non-trivial use of transfer, and at low K it beats training from scratch handily.
- Zero-shot needs a bridge between class semantics and image features. Lampert built that bridge by hand from attributes; CLIP learns it from web-scale image-text pairs.
- Composing trained foundation models, feeding the output of one into another at inference time, produces useful behaviour without any new training.

The next chapter, weak supervision, looks at what happens when you do have labels but they are noisy, partial, or generated by heuristics rather than experts. Chapter 14 picks up the composition thread again and scales it to embodied action.

## Take It from Here, Next Steps

An optional extension that deepens the lecture beats without adding new infrastructure. Work through it at your own pace after the session.

### 🦙 Run a real foundation model on your laptop

Everything above worked on a 5,000-parameter MLP backbone and an 80-parameter "text encoder". The mechanics are the published recipe, but they are easier to feel when the model on the other end is actually a real one. **Install [Ollama](https://ollama.com) and pull a small open-weights LLM** so that you have a foundation model running locally: no API key, no rate limit, no cloud round-trip. About 2 to 4 GB of disk and a few minutes are all it takes.

**Setup**

1. Download and install Ollama from [ollama.com](https://ollama.com) (Windows, macOS, and Linux installers).
2. Pull a small model that runs comfortably on a laptop CPU:
   - `ollama pull llama3.2:1b` (~1.3 GB, fastest, Meta)
   - `ollama pull phi3:mini` (~2.4 GB, strong reasoning for its size, Microsoft)
   - `ollama pull gemma2:2b` (~1.6 GB, well balanced, Google)
3. Sanity-check from a terminal: `ollama run llama3.2:1b "say hello in one word"`.

Once the Ollama daemon is running it exposes an HTTP API on `localhost:11434`, so you can drive it from Python without any extra dependency:

```python
import requests, json
resp = requests.post('http://localhost:11434/api/generate', json={
    'model': 'llama3.2:1b',
    'prompt': 'In one short sentence: what is a backbone in deep learning?',
    'stream': False,
})
print(resp.json()['response'])
```

**Exercise: in-context learning for digit attribution.** *In-context learning* is the foundation-model trick of teaching the model from examples in the prompt: no gradient updates, no fine-tuning, behaviour shaped entirely by what you put in front of it. It is a third strategy alongside the linear probe (🪜) and the contrastive bridge (🔗) for getting work out of a pretrained model.

1. **Zero-shot.** Ask the model: *"Describe the visual attributes of the digit 7 using these features: has_loop, two_loops, vertical_stroke, horizontal_stroke, curvy. Reply only with the values 0 or 1 separated by commas."* Compare the model's row to the `attrs_source[7]` we hard-coded in section 🎯.
2. **Few-shot via in-context examples.** Now provide three example rows in the prompt (digits 2, 5, 8 from `attrs_source`) before asking about a new digit. The model should latch onto the format and produce a cleaner answer.
3. **Use the LLM as the bridge in section 🔗.** Replace the linear attribute encoder W with a call that asks the LLM "is this digit description consistent with the image features I have observed?" (you will need to encode the image features as a short text). The setup is hacky on purpose; the point is to feel that the LLM *is* a foundation model and slots into the same compositional pattern as CLIP did above.

**What you will notice.** Even a 1B-parameter model, three orders of magnitude smaller than the public hosted ones, produces coherent, structured answers. That is the foundation-model recipe paying off in your hands: a backbone trained on enough breadth that it generalises to a task it was never specifically built for, addressed entirely through the prompt. Same arc as the rest of this notebook, with one of the actual published artifacts on the other end of the wire.